# Diagnostics — variance, reliability, and drift alignment

**Run this between QC and discovery.** It characterizes the featureset *before* any model is fit and
answers one question four ways: **is recalibration well-posed on this data?**

1. **Variance hierarchy** — how each feature's (and the response's) variance splits into
   **between-sensor** (structural / fabrication), **within-sensor-temporal** (drift — the
   recalibration signal), and **within-concentration** (dose-response) components; plus the raw
   feature/response distributions, an interactive 3D view, and the top drift-carrying featuretypes.
2. **Reliability (Assumption 5)** — is the modeled variance real signal or measurement noise? — both
   the whole **response** and, per **feature**, whether its **drift** sits above the replicate-noise
   floor (`drift_snr` / `R_drift`).
3. **Drift alignment** — the decisive screen: does *any* feature's drift actually **track the
   response's drift**? If nothing aligns, no architecture can recalibrate.
4. **Verdict** — reads it all back, plus a forward-chained-CV feasibility check.

> **QC is applied, not re-run.** The featureset below is **post-QC**: `extract_dataset` applies the
> gates (EIS.1–3, dose-monotonicity) using the **parameters loaded from the shared config**
> (`electropycal_analysis_config.json`) as it builds each row, so nothing here re-derives QC. Choosing
> QC stringency — the gate-impact bars and the `monotonic_r_min` sweep — now lives in
> **`quality_filtering_dashboard`** (where you save the shared config), so this notebook stays a
> characterization of the data you've *already* accepted. §2's per-feature reliability re-parses the
> raw files once to get the replicate spread (cached).

---

**Where this fits — the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. `raw_spectra_review` — inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency
2. `quality_filtering_dashboard` — set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config
3. `stabilization_review` — confirm each channel's FSCV response stabilized (optional / parallel)
4. ➡️ **`diagnostics_review` — characterize the featureset before modeling — *is recalibration well-posed?*  ← you are here**
5. `discovery_checkpointed` — run the model-discovery task queue batch-by-batch under nested CV
6. `discovery_results_review` — inspect a finished discovery run; pick a model
7. `deployment_domain_shift` — freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup — run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive


## Setup — imports, `ROOT`, shared config
Point `ROOT` at your raw PSTrace directory (or leave `None` for a synthetic demo). Band / peak / QC
parameters come from the **shared config** so this matches `quality_filtering_dashboard` and
`discovery_checkpointed`. Features are extracted **raw** (not D0-normalized) so the *structural*
component is visible — D0-normalization is exactly what removes it, leaving the drift the model
must learn.

In [ ]:
%matplotlib inline
import tempfile, math, hashlib, json
# Pin BLAS/OpenMP to 1 thread BEFORE importing numpy so parallel extraction workers
# do not oversubscribe the cores.
import os
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path
from electropycal.viz import set_pub_style
from electropycal.analysis_config import load_analysis_config, resolve_band
from electropycal.discovery.config import RESERVED_COLUMNS
from electropycal.features.extract import extract_dataset, FEATURE_SCHEMA_VERSION
from electropycal.diagreview import DiagnosticsReview
set_pub_style()
N_JOBS = 4      # worker processes for parallel feature extraction (-1 = all cores; lower if RAM-limited)

ROOT = None   # <- your own PSTrace export directory; None uses the shipped demo
# demo_input() locates demo/ from ANY working directory -- the repo root, notebooks/,
# or anywhere else -- and synthesizes an equivalent tree when no checkout is reachable
# (e.g. Colab). A relative literal here breaks in whichever cwd it was not written for.
from electropycal._demo import demo_input
ROOT = demo_input("in_vitro", ROOT)
print("ROOT =", ROOT)

CFG = load_analysis_config(ROOT)
BAND = resolve_band(ROOT, CFG)
print(f"ROOT = {ROOT}")
print(f"BAND = ({BAND[0]:.0f}, {BAND[1]:.0f}) Hz  |  peak={CFG.peak_method}  acceptance={CFG.acceptance}  "
      f"mono_tol={CFG.mono_tol}  min_snr={CFG.min_norm_snr}  r_min={CFG.monotonic_r_min}  max_reps={CFG.max_reps}")
print(f"feature-schema v{FEATURE_SCHEMA_VERSION}  (cache auto-invalidates when new predictors/targets are added)")

### Available data
A **fast** filename/header scan (no waveform parsing) of every **signal** session at `ROOT`: one row
per device with its type, id, and the size of the data it contributes —
`n_channeltimepoints` (Σ channels over its timepoints, from the 0 nM FSCV headers), `n_timepoints`,
and `n_concentrations`. Use it to decide what to select below.

In [4]:
import re as _re, pathlib as _pl
from collections import defaultdict as _dd
from electropycal.data.pstrace import parse_folder, parse_filename
_CHRE = _re.compile(r"Channel\s+(\d+)")

def _fscv_channels_fast(path):
    """Channels from an FSCV file's header line only (fast; no waveform parse)."""
    if path is None:
        return []
    for enc in ("utf-16", "utf-8"):
        try:
            with open(path, encoding=enc) as f:
                for _ in range(30):
                    ln = f.readline()
                    if not ln:
                        break
                    if "Fast Cyclic" in ln:
                        return sorted({int(x) for x in _CHRE.findall(ln)})
            return []
        except (UnicodeError, UnicodeDecodeError):
            continue
    return []

_dates, DTYPE_OF, _ch0, _conc = _dd(set), {}, {}, _dd(set)
for _folder in sorted(p for p in _pl.Path(ROOT).iterdir() if p.is_dir()):
    _fm = parse_folder(_folder.name)
    if not _fm or _fm.get("testtype") != "signal":
        continue
    for _csv in sorted(_folder.glob("*.csv")):
        _nm = parse_filename(_csv.name)
        if not _nm:
            continue
        _dev = _nm["deviceid"]; DTYPE_OF[_dev] = _fm.get("devicetype", "?"); _dates[_dev].add(_fm["date"])
        if _nm["signaltype"] == "fscv":
            _d = _nm["dose"]
            if isinstance(_d, (int, float)) and _d > 0:
                _conc[_dev].add(float(_d))
            if _d == 0.0:
                _ch0[(_dev, _fm["date"])] = _fscv_channels_fast(str(_csv))
_d0 = {dev: min(ds) for dev, ds in _dates.items()}
available = pd.DataFrame([
    {"devicetype": DTYPE_OF[dev], "deviceid": dev,
     "n_channeltimepoints": sum(len(_ch0.get((dev, d), [])) for d in _dates[dev]),
     "n_timepoints": len(_dates[dev]), "n_concentrations": len(_conc.get(dev, ()))}
    for dev in sorted(_dates)]).sort_values(["devicetype", "deviceid"]).reset_index(drop=True)

# option lists for the selector cell
AVAIL_DEVICETYPES = sorted(set(DTYPE_OF.values()))
AVAIL_DEVICES     = sorted(_dates)
AVAIL_TIMEPOINTS  = sorted({(d - _d0[dev]).days for dev in _dates for d in _dates[dev]})
AVAIL_CHANNELS    = sorted({c for chs in _ch0.values() for c in chs})
AVAIL_CONCS       = sorted({c for cs in _conc.values() for c in cs})
print(f"{len(available)} device(s)  |  devicetypes={AVAIL_DEVICETYPES}  timepoints(days)={AVAIL_TIMEPOINTS}")
print(f"channels={AVAIL_CHANNELS}  concentrations(nM)={[int(c) for c in AVAIL_CONCS]}")
display(available)

8 device(s)  |  devicetypes=['cfme', 'neurostring']  timepoints(days)=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 18, 19, 21, 25, 28, 32, 39]
channels=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16]  concentrations(nM)=[100, 250, 500, 1000, 5000]


,devicetype,deviceid,n_channeltimepoints,n_timepoints,n_concentrations
0,cfme,1-6,22,4,5
1,neurostring,2-2,109,9,5
2,neurostring,2-3,99,9,5
3,neurostring,3-1,40,5,5
4,neurostring,3-2,57,10,5
5,neurostring,3-3,33,4,5
6,neurostring,3-4,32,4,5
7,neurostring,4-2,66,10,5


### Data selector
Choose what to characterize. Each selector defaults to **`None` = all available** (devicetypes,
deviceids, channels, timepoints, concentrations). Edit and re-run — the extract cell below filters
the (cached) featureset to this selection; changing the selection is instant (no re-extraction).
This cell prints exactly the **data and config** the rest of the notebook uses.

In [5]:
# --- data selection (None = all available) ---
DEVICE_TYPES   = ["neurostring"]    # e.g. ["neurostring"] to exclude cfme; None = all
DEVICES        = None    # e.g. ["3-2", "4-2"]; None = all deviceids
CHANNELS       = None    # e.g. [1, 3, 4]; None = all channels
TIMEPOINTS     = None    # day ints e.g. [0, 1, 4]; None = all
CONCENTRATIONS = None    # nM e.g. [100, 1000]; None = all

def _shown(sel, allv): return ("all " + str(allv)) if sel is None else sel
# effective deviceids = those surviving BOTH the devicetype and deviceid filters (what the
# featureset is actually subset to). The plain "deviceids" line lists all AVAILABLE ids regardless
# of type; this line is the ones kept after the devicetype filter, so ["neurostring"] visibly drops
# the cfme ids. (Extraction still parses every device at ROOT; the selection is applied afterwards.)
_eff_devs = [d for d in AVAIL_DEVICES
             if (DEVICE_TYPES is None or DTYPE_OF.get(d) in DEVICE_TYPES)
             and (DEVICES is None or d in DEVICES)]
print("SELECTION  (axes are ANDed; the featureset is subset to their intersection)")
print(f"  devicetypes    = {_shown(DEVICE_TYPES, AVAIL_DEVICETYPES)}")
print(f"  deviceids      = {_shown(DEVICES, AVAIL_DEVICES)}")
print(f"    -> kept after devicetype+id filters: {_eff_devs}  ({len(_eff_devs)} of {len(AVAIL_DEVICES)})")
print(f"  channels       = {_shown(CHANNELS, AVAIL_CHANNELS)}")
print(f"  timepoints     = {_shown(TIMEPOINTS, AVAIL_TIMEPOINTS)}")
print(f"  concentrations = {_shown(CONCENTRATIONS, [int(c) for c in AVAIL_CONCS])}")
print("CONFIG (shared electropycal_analysis_config.json)")
print(f"  BAND=({BAND[0]:.0f}, {BAND[1]:.0f}) Hz  peak={CFG.peak_method}  acceptance={CFG.acceptance}  "
      f"mono_tol={CFG.mono_tol}  min_snr={CFG.min_norm_snr}  r_min={CFG.monotonic_r_min}  max_reps={CFG.max_reps}")

SELECTION  (axes are ANDed; the featureset is subset to their intersection)
  devicetypes    = ['neurostring']
  deviceids      = all ['1-6', '2-2', '2-3', '3-1', '3-2', '3-3', '3-4', '4-2']
    -> kept after devicetype+id filters: ['2-2', '2-3', '3-1', '3-2', '3-3', '3-4', '4-2']  (7 of 8)
  channels       = all [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16]
  timepoints     = all [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 18, 19, 21, 25, 28, 32, 39]
  concentrations = all [100, 250, 500, 1000, 5000]
CONFIG (shared electropycal_analysis_config.json)
  BAND=(2, 2000) Hz  peak=direct  acceptance=monotonic  mono_tol=0.1  min_snr=3.0  r_min=0.6  max_reps=3


### Extract featureset (cached, with ETA) → apply selection
Feature extraction is a single sequential pass — the library's parallelism (joblib fold-parallelism)
lives in **discovery**, not here — and its cost is the UTF-16 PSTrace parse per file. So this cell
does two things to stay fast on re-runs:

- **`progress=True`** prints per-session `[HH:MM:SS]` lines with a running **ETA**, so a long extract
  is monitorable.
- The full post-QC featureset is **cached** to `<ROOT>/diagnostics_featureset__<hash>.parquet` (hash
  of the extraction params). Re-running reloads it instantly; it only re-extracts when a config
  parameter changes (or you set `FORCE_REEXTRACT = True`).

The cached featureset is the whole `ROOT`; the **selection** above is then applied in-memory, so
changing devices/channels/timepoints/concentrations is instant.

In [ ]:
FORCE_REEXTRACT = False
_key = hashlib.md5(json.dumps({"band": [float(BAND[0]), float(BAND[1])], "peak": CFG.peak_method,
    "acc": CFG.acceptance, "mono_tol": CFG.mono_tol, "snr": CFG.min_norm_snr,
    "rmin": CFG.monotonic_r_min, "reps": CFG.max_reps, "d0": False, "schema": FEATURE_SCHEMA_VERSION}, sort_keys=True).encode()).hexdigest()[:8]
CACHE = Path(ROOT) / f"diagnostics_featureset__{_key}.parquet"
if CACHE.exists() and not FORCE_REEXTRACT:
    FEAT_ALL = pd.read_parquet(CACHE); print(f"loaded cached featureset: {CACHE.name}  ({len(FEAT_ALL)} rows)")
else:
    FEAT_ALL = extract_dataset(ROOT, band=BAND, peak_method=CFG.peak_method, acceptance=CFG.acceptance,
                               mono_tol=CFG.mono_tol, min_norm_snr=CFG.min_norm_snr,
                               monotonic_r_min=CFG.monotonic_r_min, max_reps=CFG.max_reps,
                               d0_normalize=False, n_jobs=N_JOBS, progress=True)   # RAW features (structure visible)
    try: FEAT_ALL.to_parquet(CACHE); print(f"cached -> {CACHE.name}")
    except Exception as e: print(f"(cache write skipped: {e})")

FEAT_ALL["devicetype"] = FEAT_ALL["device"].map(DTYPE_OF)
_sel = FEAT_ALL
if DEVICE_TYPES is not None:   _sel = _sel[_sel.devicetype.isin(DEVICE_TYPES)]
if DEVICES is not None:        _sel = _sel[_sel.device.isin(DEVICES)]
if CHANNELS is not None:       _sel = _sel[_sel.channel.isin(CHANNELS)]
if TIMEPOINTS is not None:     _sel = _sel[_sel.timepoint.isin([float(t) for t in TIMEPOINTS])]
if CONCENTRATIONS is not None: _sel = _sel[_sel.concentration.isin([float(c) for c in CONCENTRATIONS])]
FEAT = _sel.reset_index(drop=True)

# The review object computes the feature-type taxonomy + per-feature variance decomposition once;
# every section below is a thin call on it. Rebuild to customize (e.g. a different selection).
DR = DiagnosticsReview(FEAT, cfg=CFG, root=ROOT, band=BAND, n_jobs=N_JOBS)
DR.print_settings()

### Feature dictionary — definitions & derivations
What each feature type *is* and how it is derived, its units, its role (**predictor** vs **RESERVED
target** — DA-derived targets are never predictors, to keep the model leakage-free), and its
D0-normalization kind (additive shift vs multiplicative ratio). Per-frequency EIS types (`R_s`, `C_s`,
…) are emitted as `<type>_fNN` across the in-band frequency grid; `_integral` / `_band_*` / `_ratio`
are whole-spectrum summaries. Built from the actual columns in this featureset.

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(DR.feature_dictionary())
print("Predictors feed the model; DA-derived TARGET types are RESERVED (leakage-free); metadata "
      "describe the row. Also: electropycal.print_feature_catalog().")

## 1. Variance hierarchy

This section reads **1.1** raw feature/response distributions → **1.2** featuretypes ranked by drift → **1.3** the full variance decomposition → **1.4** an interactive 3D view. The decomposition (used in 1.2–1.3) splits each feature's (and the response's) total variance across all valid rows into three components that sum to 1:

- **between-sensor** — sensor-to-sensor / fabrication differences (a *fixed* offset per sensor).
  **D0-normalization removes this**; it is not drift.
- **within-time** — variation across timepoints *within* a sensor: **electrode drift, the
  recalibration signal we want to model.**
- **within-dose** — the dose-response (for the response this is the calibration signal; for the EIS
  features, measured once per channel-timepoint, it is ~0).

**How the split is computed (`diagnostics.variance.variance_hierarchy`).** A **sequential
sum-of-squares (hierarchical nested-ANOVA) partition**, *not* a full variance-components model:
`SS_total = Σ(v − v̄)²` is peeled in order — `SS_between` from the sensor means, then `SS_within-time`
from the sensor×timepoint means (after removing the sensor mean), and the remainder is `SS_within-dose`.
Reported as fractions of `SS_total`. **Caveats:** it is a *descriptive, ordered* decomposition on
**unbalanced** groups (unequal channels/timepoints/doses), so — unlike a balanced-design REML/ICC
variance-components model (VCA) — the components are order-dependent and not unbiased population
variance components; read them as "share of observed spread", not as an ICC. It is exactly the
sequential `SS` (sum-of-squares) decomposition.

### 1.1 Featuretype distributions
The raw value distributions. First the **response** `NormIpeak` (red), then one panel per
**frequency-independent** featuretype (global EIS descriptors + FSCV predictors `mean_Vpeak` /
`mean_Ibg`), then one panel per **frequency-dependent** featuretype as a **density (violin) profile per
in-band frequency** — the rotated profiles show *how the distribution shape* changes with frequency
(e.g. bimodal → unimodal), which overlaid histograms/color-fades cannot. Violin median = red.

Every histogram marks **mean (dashed blue)**, **median (dashed red)** and a **±1 s.d. band**, legend
ordered `n / mean / median / s.d.` The **sample unit differs by feature** (shown as the legend `n`):
EIS features are one value per **channel-timepoint** (`n_chtimepoints`, dose-invariant), while the
response and FSCV predictors vary per **channel-timepoint-concentration** (`n_samples`).

In [ ]:
DR.plot_featuretype_distributions()

### 1.2 Top featuretypes by within-time (drift) variance
Featuretypes aggregated (frequency-dependent types averaged over their frequency columns), then
**ranked by within-time (drift) variance**. The stacked bars show the variance breakdown (**within-time
at the base**, so the drift regions are bottom-aligned and directly comparable across featuretypes) for
every featuretype whose within-time share is **above the median** — the ones that actually track
degradation — with the **response as the first bar (red outline + red label)** for reference. The table
lists the **top 10 featuretypes by within-time variance**.

In [ ]:
print("top 10 featuretypes by within-time (drift) variance share:")
with pd.option_context("display.max_rows", None):
    display(DR.plot_variance_breakdown().round(3))

### 1.3 Overall featureset variance breakdown
Across **all features** (response excluded), the distribution of each variance component — one point per
**feature** (`n_features`). Panels: **between-sensor (grey)**, **within-time (blue)**, **within-dose
(purple)**; **mean = dashed blue**, **median = dashed red**, shaded **±1 s.d.** band (values in the
legend). The table repeats the per-component statistics over the feature set.

In [ ]:
print("per-component statistics across the feature set (fractions):")
with pd.option_context("display.max_rows", None):
    display(DR.plot_variance_decomposition().round(3))

### 1.4 Feature point cloud — interactive 3D
The selected feature (default: the response `NormIpeak`) as an **interactive** point cloud over
**channel × timepoint × concentration**, colored by feature magnitude on a **sequential gradient in
that featuretype's own color** (light → saturated). The **concentration axis is log-scaled** (doses are
geometric). Rotate/zoom to see how a feature moves with time (drift) and dose; set `FEATURE_3D` to any
column. *(Interactive via plotly, which Colab ships; locally `pip install "electropycal[notebooks]"` —
else a static matplotlib fallback is drawn.)*

> EIS features are constant along the concentration axis (one value per channel-timepoint); the response
> and FSCV predictors vary with dose.

In [ ]:
FEATURE_3D = "NormIpeak"          # response (default) or any feature column, e.g. "C_s_f00"
COLOR_BY_CHANNEL = False          # (kept for API parity; the folded plot uses one gradient)
DR.plot_feature_3d(FEATURE_3D, color_by_channel=COLOR_BY_CHANNEL)

### 1.5 The response is a ratio — NormIpeak numerator vs denominator drift
`NormIpeak = (I_sig − I_bg) / I_bg` at V_ox = **(background-subtracted faradaic peak) ÷ background**.
Two *independent* physical quantities move it, so a NormIpeak change alone does **not** say whether the
sensor's *faradaic sensitivity* changed or its *background / interfacial capacitance* changed:

| background (denom.) | peak (numer.) | bg-subtracted Ipeak | NormIpeak = peak/bg |
|---|---|---|---|
| stable | ↑ / ↓ | ↑ / ↓ | ↑ / ↓ (tracks peak) |
| ↑ | stable | ~0 | **↓** (bg dilutes) |
| ↓ | stable | ~0 | **↑** (bg concentrates) |
| ↑ | ↑ | ↑ | ratio (ambiguous) |
| ↓ | ↓ | ↓ | ratio (ambiguous) |
| ↑ | ↓ | ↓↓ | ↓↓ |
| ↓ | ↑ | ↑↑ | ↑↑ |

**NormIpeak goes negative only when the numerator does** (`I_sig < I_bg` at V_ox — a non-physical
oxidation peak), which extraction drops (`drop_negative`); the denominator only *scales* it. This
numerator/denominator ambiguity is exactly why the **EIS features** — which independently measure the
interface (`C_s`, `C_p`, `R_s`) — are the predictors: they let the model attribute NormIpeak drift to
its physical cause. It also motivates the **sensitivity / calibration-curve** target framing, since a
per-dose NormIpeak *magnitude* conflates faradaic-sensitivity drift with background drift.

## 2. Reliability (Assumption 5) — is the modeled variance signal or noise?
Four questions, a **2×2** of {response, feature} × {measurement, drift}. "Measurement" asks whether a
quantity is resolved above cycle-to-cycle replicate noise *at all* (total spread vs noise); "drift"
asks whether its **within-time** slice — the part recalibration models — is above that same noise
floor. All noise floors come from the replicate spread (`σ²_meas / n_rep`).

1. **2.1** response measurement · **2.2** response drift · **2.3** feature measurement · **2.4** feature drift.

### 2.1 Is the response measured above its noise floor?
**Q.** Is `NormIpeak` resolved above cycle noise at all? **Metric.** `R = σ²_signal/σ²_obs ∈ [0,1]`,
`σ²_signal = σ²_obs − σ²_meas/n_rep`. **How.** `σ²_obs` = total variance of NormIpeak across
observations; `σ²_meas` = mean of `rep_std²` (replicate noise). **Caveat.** `σ²_obs` is dose-dominated
(§2.2 isolates the drift). **Left:** σ²_obs split into signal + noise. **Right:** per-observation
replicate variance vs the total spread.

In [ ]:
DR.plot_response_reliability()   # sets DR.R / DR.sigma2_meas (used by the drift + verdict sections)

### 2.2 Is the response *drift* above its noise floor?
**Q.** Is the temporal (within-time) drift of `NormIpeak` above cycle noise? **Metric.** `R_drift`,
`drift_snr = σ²_drift / (σ²_meas/n_rep)` where `σ²_drift` is the within-time variance component.
**Caveat — dose-confounded.** `NormIpeak`'s variance is dominated by the dose-response, so its
"within-time" slice is the *dose-averaged level* shifting over time, entangled with any change in the
dose-response itself — it is **not** a clean drift the way an EIS feature's is (§2.4). The plot shows
the response's absolute variance decomposition with the replicate-noise floor marked.

In [ ]:
DR.plot_response_drift_reliability()

### 2.3 Are the feature measurements above their noise floors?
**Q.** Is each predictor feature resolved above cycle noise at all (before asking about drift)?
**Metric.** per-feature `R_meas = (σ²_obs − σ²_meas/n_rep)/σ²_obs`. **How.** `σ²_meas` from the
per-replicate re-extraction (`replicate_feature_reliability`, cached); `σ²_obs` = the feature's
variance over its natural observations (channel-timepoints for dose-invariant EIS features, samples
for FSCV predictors). **Left:** `R_meas` distribution. **Right:** `σ²_obs` vs the noise floor (log–log;
above the diagonal = the feature carries real between-observation signal, not just noise).

In [ ]:
DR.plot_feature_measurement_reliability()

### 2.4 Are the feature *drifts* above their noise floors?
**Q.** For each feature, is its **within-time (drift)** variance above its measurement-noise floor —
the clean per-feature version of §2.2 (EIS features are dose-invariant, so within-time *is* the drift).
**Metric.** `R_drift`, `drift_snr = σ²_drift / (σ²_meas/n_rep)`. **Left:** `R_drift` distribution.
**Right:** `σ²_drift` vs `noise_floor` (log–log; above the line = resolvable), colored by featuretype.
**Caveat.** Few timepoints upward-bias `σ²_drift`, so read `drift_snr > 1` as "not obviously
noise-dominated"; it sharpens with more timepoints.

**Each plotted point is one *feature*** — `R_drift` and `σ²_drift` come from a **variance decomposition** of that feature over *all* rows (the within-time / drift share and its replicate-noise floor), not a mean of data points; the §1.1 histograms, by contrast, plot one value per sample.

In [ ]:
with pd.option_context("display.max_rows", None):
    display(DR.plot_feature_drift_reliability().round(3))

## 3. Drift alignment — does any feature's drift track the response's?
**Question.** The decisive pre-discovery screen: does *any* feature's drift **co-move with the
response's drift**? A feature can drift cleanly (§2.2) yet be irrelevant to `NormIpeak`.

**Metric & why it fits.** A within-`(sensor, concentration)` **temporal Pearson `r`** between feature
and response: at a fixed sensor and dose, as time passes, do the two move together? Holding sensor and
dose fixed isolates the *drift* co-movement — exactly what a recalibration model exploits. Reported
**signed**: a feature drifting opposite the response (`r < 0`) is as usable as one drifting with it
(`r > 0`) — it is `|r|` that measures tracking strength.

**How it's computed.** Per `(sensor, concentration)` cell (≥3 timepoints, nonzero variance) the
timepoint-mean feature and response are correlated; the per-cell `r`s are pooled by Fisher-`z`
(`drift_alignment`). **Left:** the signed-alignment distribution. **Right:** the **feature-quality map**
— `R_drift` (is the drift real?) vs signed alignment (does it track?), colored by featuretype; points
far from `alignment = 0` *and* high in `R_drift` are the recalibration signal.

**Caveats.** Few timepoints per cell ⇒ each `r` is noisy (pooling across cells stabilizes it); this is a
**univariate** screen — discovery's multivariate model can still combine weakly-aligned features, so
treat low `|alignment|` as a warning, not a veto, unless *nothing* aligns.

**Each plotted point is one *feature*** (e.g. `C_s_f03`), not a data point or a mean of data points: `alignment_r` is the Fisher-z-pooled per-`(sensor,concentration)` temporal correlation described above; the map's x = `R_drift`, y = signed `alignment_r`, **size** = within-time variance share, **color** = featuretype. (The only averaging is collapsing replicate rows at the same timepoint within a cell before correlating.)

In [ ]:
with pd.option_context("display.max_rows", None):
    display(DR.plot_drift_alignment().round(3))

### 3.1 Frequency-bandwidth tradeoff — where does aligned drift live across the full spectrum?

The band does two jobs at once: any in-band inductive point **fails** a channel-timepoint (EIS.1), and
the same band is the window features are extracted over. So the choice of ceiling trades feature
**bandwidth** against **retention**. This weighs both across the **entire measured frequency range**
(not just the current band):

- **left — retention:** channel-timepoints that stay capacitive vs the upper band edge (from one raw
  EIS pass). Raising the ceiling can only drop, never add.
- **right — signal:** **signed** drift-alignment `r` (feature drift vs response drift) at **every
  measured frequency**, one trace per frequency-dependent featuretype. Features are re-derived over the
  full grid for the **current-valid** channel-timepoints (no re-gating), so you can see whether
  frequencies **above the current ceiling** carry aligned drift worth widening the band for.

The current band is shaded; its **lower edge is green dashed, upper edge red dashed**. **Rule:** widen
the ceiling only if aligned drift (right, away from 0) extends past it and enough channels survive there
(left). *(One raw EIS scan — the slow cell.)*

In [ ]:
DR.plot_frequency_bandwidth_tradeoff()

### 3.2 Sanity checks — how the alignment diagnostics are computed, and feature/response correlations
Two alignment metrics, **both Pearson r ∈ [−1, 1] so their magnitudes are directly comparable** (same
scale — a larger |r| is a stronger relationship regardless of which metric):
- **Drift alignment** (§3; what discovery trains on with D0 features): within each `(sensor,
  concentration)` cell, correlate the feature's **time series** vs the response's across timepoints —
  *as one sensor drifts at a fixed dose, do they co-move?* Pooled across cells by Fisher-z. This
  **differences out** each sensor's absolute level (and amplifies replicate noise).
- **Absolute (cross-sectional) alignment**: within each concentration, correlate the feature's
  **absolute level** vs the response's, pooled across **all sensor-timepoints** — *do high-feature
  states have high response, across sensors?* This is the literal state→sensitivity hypothesis, and the
  level that D0/drift remove. If |absolute| ≫ |drift|, the signal is in the level (test
  `d0_normalize=False`, validate on Track 3); if |drift| ≫ |absolute| the change is the signal and the
  D0 framing is right.

The **cross-correlation heatmap** is built at the **per-(sensor, timepoint)** level, **pooled across all
sensors and timepoints** (cross-sectional). It shows the top predictors (by |drift alignment|) against
**every response variable** — dose-resolved `NormIpeak` (one column per concentration, *not* dose-pooled),
the faradaic peak shape (`peak_height/area/fwhm`), and the curve-fit params (`sensitivity`, intercept,
curvature, `power_beta`, `sat_imax`, `hill_imax`). Dense off-diagonal among predictors = collinear EIS
(why PLS/selection matter); the predictor×response block is the cross-sectional signal.

In [ ]:
DR.plot_alignment_sanity_checks()

## 4. Verdict — is recalibration well-posed here?
Reads §1–§3 back so the decision is explicit, and adds a **forward-chained-CV feasibility** check
(nested CV needs at least a train timepoint before each test timepoint, i.e. ≥2 timepoints per
sensor).

In [ ]:
DR.verdict()

> **Decision to record — is recalibration well-posed? (gate before discovery).** Read the verdict
> above together with §3 (drift alignment). **Look for:** at least one EIS / background predictor
> whose drift *aligns* with the response's drift (§3) **and** sits above its replicate-noise floor
> (§2.4). **Rule:** if nothing aligns above noise, **stop** — no architecture can recalibrate a
> response its predictors don't track; revisit the band / QC or the response framing first. If it *is*
> well-posed, proceed to discovery. This is decision **4** in `docs/DESIGN.md`.

## 5. D0-normalization effect — what discovery actually trains on
Everything above is on **raw** features (so the *structural* between-sensor spread is visible).
Discovery instead trains on **D0-normalized** features: each feature is referenced per
sensor to its **own earliest timepoint** — additive shift for bounded/phase/log-slope types, division
(fraction-of-baseline) for magnitude/area types. This is what removes the between-sensor structure,
leaving the drift the model learns.

**Left:** the median variance decomposition **raw vs D0-normalized** — the between-sensor share should
collapse toward 0, and the within-time (drift) + within-dose shares should rise (the same signal, now
the dominant variance). **Right:** the top-drift feature's per-sensor trajectories over time, raw vs
D0-normalized — sensors that were offset by fabrication now start from a common baseline, so their
*drift* is directly comparable. (Note: for multiplicative features D0-norm expresses drift as
fraction-of-baseline, which rescales small-baseline features — a reason drift magnitude in §3 shifts
after normalization.)

**§5b** then confirms D0-normalization is a per-sensor *affine* transform: the §3 drift↔response **alignment is invariant** (raw vs D0 on the diagonal) and the §2.4 drift reliability is preserved — only the *structural* variance share (§1) changes. So the raw-data diagnostics above read the same on what discovery actually trains on.

In [ ]:
DR.plot_d0_effect()

In [ ]:
DR.plot_d0_invariance()

## 6. Response / deformation modes — which drift, and is the drift resolvable?
`diagnostics` above analyzes only the default response `NormIpeak`. But recalibration can target *any*
of several response variables (each a candidate **evolution mode**), and the question "what modes
exist" is answered by **which of them actually drift** (within-time share) and **whether that drift is
above replicate noise** (`R_drift`). This section runs the §1/§2 diagnostics across them:

- **Per-dose responses** (in the featureset): the ratio **`NormIpeak`**, its **numerator**
  `peak_height` (bg-subtracted peak) and **denominator** `mean_Ibg` (background) — so you can see which
  side drives the ratio's drift — plus **`peak_area`** (charge) and **`peak_fwhm`** (width / kinetics).
- **Calibration-curve modes** (per sensor-timepoint, via `sensitivity_featureset`): **intercept**
  (DC level), **slope** (`sensitivity`), and **curvature** (shape). These have no dose axis and no
  per-replicate noise estimate, so only their between-sensor / within-time split is shown.

**Read:** the **within-time** bar = how much a mode drifts; the **R_drift** label = whether that drift
is real signal (a mode with large drift *and* R_drift→1 is a genuine, resolvable evolution mode). This
complements `discovery_results_review §6`, which asks which of these is actually *predictable* from EIS.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(DR.plot_response_modes().round(3))
print("Which modes DRIFT = within-time bar; whether the drift is REAL = R_drift (per-dose only). "
      "discovery_results_review §6 then tests which are PREDICTABLE from EIS.")

> **Decision to record — response framing (1) and numerator-vs-denominator (2).** From the modes
> table above: **(1) framing** — compare `NormIpeak` (per-dose ratio) against the calibration-curve
> modes (`sensitivity` slope / intercept / curvature) on *which drifts and is resolvable* (`R_drift`);
> the one that carries real, resolvable drift is your `TARGET` (confirm it is *predictable* from EIS in
> `discovery_results_review` §6). **(2) num vs denom** — see whether `NormIpeak`'s drift is driven by
> its **numerator** `peak_height` or its **denominator** `mean_Ibg`; if `mean_Ibg` both drives the
> ratio *and* is used as a predictor, decide whether to drop it (circularity, see §1.5). **Record**
> both in `docs/DESIGN.md` (decisions 1 and 2).